In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# データの読み込み
file_path = r"E:\refit\all_roi_spectra.csv"
df = pd.read_csv(file_path)

# 波長カラムの抽出と範囲設定
wave_cols = [col for col in df.columns if col.startswith('wave_')]
vnir_end_col = 'wave_975.00nm'
swir_start_col = 'wave_901.01nm'

vnir_end_idx = wave_cols.index(vnir_end_col)
swir_start_idx = wave_cols.index(swir_start_col)

vnir_cols = wave_cols[:vnir_end_idx+1]
swir_cols = wave_cols[swir_start_idx:]

def get_wavelengths(cols):
    waves = []
    for col in cols:
        match = re.search(r'wave_([\d\.]+)nm', col)
        if match:
            waves.append(float(match.group(1)))
    return np.array(waves)

vnir_waves = get_wavelengths(vnir_cols)
swir_waves = get_wavelengths(swir_cols)

# ピクセル選択
pixel_index = 1052
pixel_data = df.iloc[pixel_index]

# データ取得
vnir_spectrum = pixel_data[vnir_cols].values
swir_spectrum = pixel_data[swir_cols].values
x_coord = pixel_data['x']
y_coord = pixel_data['y']

# グラフ描画
plt.figure(figsize=(10, 6))
plt.plot(vnir_waves, vnir_spectrum, color='blue', label='VNIR', alpha=0.8)
plt.plot(swir_waves, swir_spectrum, color='red', label='SWIR', alpha=0.8)

plt.xlabel('Wavelength [nm]')
plt.ylabel(r'Radiance [$W / (m^2 \cdot sr \cdot \mu m)$]')
plt.title(f' VNIR and SWIR Spectrum (Pixel x={x_coord}, y={y_coord})')
plt.legend()
plt.grid(True)

plt.savefig('vnir_swir_overlay.png')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import matplotlib.cm as cm

# ファイルの読み込み
df_methane = pd.read_csv(r"E:\refit\water_lut\2.0.csv")
df_hisui = pd.read_csv(r"E:\メタン\2025_HISUI_1_地獄の門\HSHL1G_N402E0583_20210803092156_20220830001542\HSHL1G_N402E0583_20210803092156_20220830001542_B.csv")

# ご提示いただいた装置関数
def instrumental_function(data, sigma, mu): 
    column, row = data.shape 
    x = data[0, :] # Wavelen
    out = np.zeros((column, row))
    out[0, :] = x
    ave = np.mean(x) # 波長の平均値
    # ガウス関数 (全体グリッドに対して定義)
    gauss = np.exp(-(x - ave - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi)) 
    for i in range(1, column):
        y = data[i, :]
        out[i, :] = np.convolve(y, gauss, mode="same") # 畳み込み
    return out

# --- データの前処理 ---
# df_methaneを転置して、行方向を波長・各濃度データにする (instrumental_functionの入力形式に合わせる)
# 行0: 波長, 行1~: 各濃度のスペクトル
data = df_methane.T.values

# --- HISUIのパラメータ設定 ---
# SWIR帯域(>1000nm)の半値幅(FWHM)の平均を取得
swir_bands = df_hisui[df_hisui['CenterWavelengthNanometer'] > 1000]
avg_fwhm = swir_bands['FullWidthAtHalfMaximumNanometer'].mean()
print(f"HISUI SWIR Average FWHM: {avg_fwhm:.4f} nm")

# FWHM = 2.355 * sigma より sigma を算出
sigma = avg_fwhm / 2.355
mu = 0 # 中心はずらさない

# --- 装置関数の適用 ---
smoothed_data = instrumental_function(data, sigma, mu)

# --- HISUI波長へのリサンプリング ---
# HISUIの中心波長を取得
hisui_wavs = df_hisui['CenterWavelengthNanometer'].values
# 元データの波長範囲内にあるHISUIバンドのみ抽出
data_wavs = data[0, :]
valid_indices = (hisui_wavs >= data_wavs.min()) & (hisui_wavs <= data_wavs.max())
target_wavs = hisui_wavs[valid_indices]

resampled_rows = []
resampled_rows.append(target_wavs) # 0行目は波長

# 平滑化データ(smoothed_data)をHISUI波長(target_wavs)に補間
for i in range(1, smoothed_data.shape[0]):
    y = smoothed_data[i, :]
    # 線形補間関数を作成
    f = interp1d(data_wavs, y, kind='linear')
    resampled_rows.append(f(target_wavs))

resampled_data = np.array(resampled_rows)

# --- プロット ---
# プロットする範囲の定義
plot_ranges = [
    (1000, 2500, "HISUI SWIR Range (1000-2500nm)"),
    (1500, 1800, "Zoom: 1500-1800nm"),
    (2100, 2400, "Zoom: 2100-2400nm")
]

ppm_labels = df_methane.columns[1:] # 0, 0.25, ... 5
colors = cm.viridis(np.linspace(0, 1, len(ppm_labels))) # 色の準備

fig, axes = plt.subplots(3, 1, figsize=(12, 18))

for ax, (start, end, title) in zip(axes, plot_ranges):
    # 波長範囲でフィルタリング
    mask = (resampled_data[0, :] >= start) & (resampled_data[0, :] <= end)
    x_plot = resampled_data[0, mask]
    
    # 各濃度のプロット
    for i in range(1, resampled_data.shape[0]):
        y_plot = resampled_data[i, mask]
        ax.plot(x_plot, y_plot, color=colors[i-1], linewidth=1)
        
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Wavelength [nm]", fontsize=12)
    ax.set_ylabel("Radiance / Value", fontsize=12)
    ax.grid(True, alpha=0.5)

# カラーバーの追加
sm = plt.cm.ScalarMappable(cmap=cm.viridis, norm=plt.Normalize(vmin=0, vmax=5))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), label='Methane Concentration (ppm)')

plt.savefig('hisui_methane_simulation.png')
plt.show()

In [ ]:
# --- プロット（3つの別図として表示） ---
plot_ranges = [
    (1000, 2500, "HISUI SWIR Range "),
    (1600, 1800, "Zoom: 1600-1800nm"),
    (2100, 2400, "Zoom: 2100-2400nm"),
]

ppm_labels = df_methane.columns[1:]  # 0, 0.25, ... 5
colors = cm.viridis(np.linspace(0, 1, len(ppm_labels)))  # 元コード踏襲

figs = []

for idx, (start, end, title) in enumerate(plot_ranges, start=1):
    fig, ax = plt.subplots(figsize=(12, 6))

    # 波長範囲でフィルタリング
    mask = (resampled_data[0, :] >= start) & (resampled_data[0, :] <= end)
    x_plot = resampled_data[0, mask]

    # 各濃度のプロット
    for i in range(1, resampled_data.shape[0]):
        y_plot = resampled_data[i, mask]
        ax.plot(x_plot, y_plot, color=colors[i-1], linewidth=1)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Wavelength [nm]", fontsize=12)
    ax.set_ylabel(r'Radiance [$W / (m^2 \cdot sr \cdot \mu m)$]', fontsize=12)
    ax.grid(True, alpha=0.5)

    # 各図ごとにカラーバーを付与（0〜5 ppmに正規化）
    sm = plt.cm.ScalarMappable(cmap=cm.viridis, norm=plt.Normalize(vmin=0, vmax=5))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label='Methane Concentration (ppm)')

    fig.tight_layout()
    figs.append(fig)

# 個別に保存（不要ならコメントアウト）
for i, fig in enumerate(figs, start=1):
    fig.savefig(f"hisui_methane_simulation_{i}.svg", dpi=150)

plt.show()


In [ ]:

TITLE_FS = 22
LABEL_FS = 20
TICK_FS  = 16
CBAR_LABEL_FS = 18
CBAR_TICK_FS  = 14

for idx, (start, end, title) in enumerate(plot_ranges, start=1):
    fig, ax = plt.subplots(figsize=(12, 6))


    # 波長範囲でフィルタリング
    mask = (resampled_data[0, :] >= start) & (resampled_data[0, :] <= end)
    x_plot = resampled_data[0, mask]

    # 各濃度のプロット
    for i in range(1, resampled_data.shape[0]):
        y_plot = resampled_data[i, mask]
        ax.plot(x_plot, y_plot, color=colors[i-1], linewidth=1)

    ax.set_title(title, fontsize=TITLE_FS, fontweight="bold")
    ax.set_xlabel("Wavelength [nm]", fontsize=LABEL_FS, fontweight="bold")
    ax.set_ylabel(r'Radiance [$W / (m^2 \cdot sr \cdot \mu m)$]',
                  fontsize=LABEL_FS, fontweight="bold")

    # 追加：目盛りの文字サイズ
    ax.tick_params(axis="both", which="major", labelsize=TICK_FS)

    ax.grid(True, alpha=0.5)

    sm = plt.cm.ScalarMappable(cmap=cm.viridis, norm=plt.Normalize(vmin=0, vmax=5))
    sm.set_array([])

    # 追加：カラーバーの文字サイズ
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Methane Concentration (ppm)", fontsize=CBAR_LABEL_FS, fontweight="bold")
    cbar.ax.tick_params(labelsize=CBAR_TICK_FS)

    fig.tight_layout()
    figs.append(fig)

for i, fig in enumerate(figs, start=1):
    fig.savefig(f"hisui_methane_simulation_{i}.svg", dpi=150)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# SVGをPPTで扱いやすく（文字がパス化されにくい）
plt.rcParams["svg.fonttype"] = "none"

# --- プロット（3つの別図として表示） ---
plot_ranges = [
    (1000, 2500, "HISUI SWIR Range"),
    (1600, 1800, "Zoom: 1600–1800 nm"),
    (2100, 2400, "Zoom: 2100–2400 nm"),
]

ppm_labels = df_methane.columns[1:]   # 0, 0.25, ... 5
N = len(ppm_labels)

# === 単色グラデ設定（青系） ===
# 白に近い薄色(見づらい)を避けるため、0.35〜0.95を使用
cmap = cm.Blues
colors_list = cmap(np.linspace(0.35, 0.95, N))

# PPT向け文字サイズ
TITLE_FS = 22
LABEL_FS = 20
TICK_FS  = 16
CBAR_LABEL_FS = 18
CBAR_TICK_FS  = 14

figs = []

for idx, (start, end, title) in enumerate(plot_ranges, start=1):
    fig, ax = plt.subplots(figsize=(12, 6))

    # 波長範囲でフィルタリング
    mask = (resampled_data[0, :] >= start) & (resampled_data[0, :] <= end)
    x_plot = resampled_data[0, mask]

    # 各濃度のプロット
    for i in range(1, resampled_data.shape[0]):
        y_plot = resampled_data[i, mask]
        ax.plot(
            x_plot, y_plot,
            color=colors_list[i-1],
            linewidth=1.7,   # 少し太め
            alpha=0.95
        )

    ax.set_title(title, fontsize=TITLE_FS, fontweight="bold")
    ax.set_xlabel("Wavelength [nm]", fontsize=LABEL_FS, fontweight="bold")
    ax.set_ylabel(r'Radiance [$W / (m^2 \cdot sr \cdot \mu m)$]',
                  fontsize=LABEL_FS, fontweight="bold")
    ax.tick_params(axis="both", which="major", labelsize=TICK_FS)
    ax.grid(True, alpha=0.4)

    # カラーバー（cmapを線と統一）
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=5))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Methane Concentration (ppm)", fontsize=CBAR_LABEL_FS, fontweight="bold")
    cbar.ax.tick_params(labelsize=CBAR_TICK_FS)

    fig.tight_layout()
    figs.append(fig)

# 個別に保存（SVG推奨）
for i, fig in enumerate(figs, start=1):
    fig.savefig(f"hisui_methane_simulation_{i}_blues.svg")
    fig.savefig(f"hisui_methane_simulation_{i}_blues.png", dpi=300)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors

plt.rcParams["svg.fonttype"] = "none"  # PPTで編集しやすい

plot_ranges = [
    (1000, 2500, "HISUI SWIR Range"),
    (1600, 1800, "Zoom: 1600–1800 nm"),
    (2100, 2400, "Zoom: 2100–2400 nm"),
]

ppm_labels = df_methane.columns[1:]
N = len(ppm_labels)

# ===== ここが配色の肝：viridisの黄色側を捨てる =====
base = cm.viridis
cmap = colors.LinearSegmentedColormap.from_list(
    "viridis_no_yellow",
    base(np.linspace(0.05, 0.72, 256))  # 0.72くらいまでだと黄色がほぼ出ない
)
colors_list = cmap(np.linspace(0, 1, N))

# PPT向け文字サイズ（必要なら調整）
TITLE_FS = 22
LABEL_FS = 20
TICK_FS  = 16
CBAR_LABEL_FS = 18
CBAR_TICK_FS  = 14

figs = []

for idx, (start, end, title) in enumerate(plot_ranges, start=1):
    fig, ax = plt.subplots(figsize=(12, 6))

    mask = (resampled_data[0, :] >= start) & (resampled_data[0, :] <= end)
    x_plot = resampled_data[0, mask]

    for i in range(1, resampled_data.shape[0]):
        y_plot = resampled_data[i, mask]
        ax.plot(
            x_plot, y_plot,
            color=colors_list[i-1],
            linewidth=1.8, alpha=0.95
        )

    ax.set_title(title, fontsize=TITLE_FS, fontweight="bold")
    ax.set_xlabel("Wavelength [nm]", fontsize=LABEL_FS, fontweight="bold")
    ax.set_ylabel(r'Radiance [$W / (m^2 \cdot sr \cdot \mu m)$]',
                  fontsize=LABEL_FS, fontweight="bold")
    ax.tick_params(axis="both", which="major", labelsize=TICK_FS)
    ax.grid(True, alpha=0.4)

    # カラーバーも同じcmap
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=5))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Methane Concentration (ppm)", fontsize=CBAR_LABEL_FS, fontweight="bold")
    cbar.ax.tick_params(labelsize=CBAR_TICK_FS)

    fig.tight_layout()
    figs.append(fig)

for i, fig in enumerate(figs, start=1):
    fig.savefig(f"hisui_methane_simulation_{i}_no_yellow.svg")
    fig.savefig(f"hisui_methane_simulation_{i}_no_yellow.png", dpi=300)

plt.show()


In [3]:
# 実行前にライブラリのインストールが必要です: pip install graphviz
from graphviz import Digraph

def create_flowchart():
    # フォーマットを 'png' または 'svg' に指定
    dot = Digraph('methane_flowchart', format='svg')
    dot.attr(rankdir='LR')  # Left to Right (横長)
    dot.attr('node', fontname='Meiryo') # Windowsの場合。環境に合わせてフォントを変更してください

    # ノードの定義
    # Data nodes
    dot.attr('node', shape='parallelogram', style='filled', fillcolor='#ffe6e6')
    dot.node('input', '観測スペクトル\n(L1Gデータ)')
    dot.node('output', '推定結果\n(CH4濃度)')

    # Process nodes
    dot.attr('node', shape='box', style='rounded,filled', fillcolor='#e6f3ff')
    dot.node('step1', 'Step 1: H2O推定\n(1025-1238nm)\nσ固定')
    dot.node('lut', 'LUT選択\n最適LUTをロード')
    dot.node('step2', 'Step 2: σ予備推定\nCH4初期値固定\nσ最適化')
    dot.node('step3', 'Step 3: CH4最終推定\n濃度推定')

    # エッジの定義
    dot.edge('input', 'step1')
    dot.edge('step1', 'lut')
    dot.edge('lut', 'step2')
    dot.edge('step2', 'step3')
    dot.edge('step3', 'output')

    # 出力 (カレントディレクトリに methane_flowchart.svg が生成されます)
    dot.render(view=True)

if __name__ == '__main__':
    try:
        create_flowchart()
        print("フローチャートを作成しました。")
    except Exception as e:
        print(f"エラーが発生しました: {e}\nGraphvizがインストールされているか確認してください。")

ModuleNotFoundError: No module named 'graphviz'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import matplotlib.cm as cm

# ファイルの読み込み
df_methane = pd.read_csv(r"E:\refit\waterprox\modtranwaterdata_full.csv")
df_hisui = pd.read_csv(r"E:\メタン\2025_HISUI_1_地獄の門\HSHL1G_N402E0583_20210803092156_20220830001542\HSHL1G_N402E0583_20210803092156_20220830001542_B.csv")

# ご提示いただいた装置関数
def instrumental_function(data, sigma, mu): 
    column, row = data.shape 
    x = data[0, :] # Wavelen
    out = np.zeros((column, row))
    out[0, :] = x
    ave = np.mean(x) # 波長の平均値
    # ガウス関数 (全体グリッドに対して定義)
    gauss = np.exp(-(x - ave - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi)) 
    for i in range(1, column):
        y = data[i, :]
        out[i, :] = np.convolve(y, gauss, mode="same") # 畳み込み
    return out

# --- データの前処理 ---
# df_methaneを転置して、行方向を波長・各濃度データにする (instrumental_functionの入力形式に合わせる)
# 行0: 波長, 行1~: 各濃度のスペクトル
data = df_methane.T.values

# --- HISUIのパラメータ設定 ---
# SWIR帯域(>1000nm)の半値幅(FWHM)の平均を取得
swir_bands = df_hisui[df_hisui['CenterWavelengthNanometer'] > 1000]
avg_fwhm = swir_bands['FullWidthAtHalfMaximumNanometer'].mean()
print(f"HISUI SWIR Average FWHM: {avg_fwhm:.4f} nm")

# FWHM = 2.355 * sigma より sigma を算出
sigma = avg_fwhm / 2.355
mu = 0 # 中心はずらさない

# --- 装置関数の適用 ---
smoothed_data = instrumental_function(data, sigma, mu)

# --- HISUI波長へのリサンプリング ---
# HISUIの中心波長を取得
hisui_wavs = df_hisui['CenterWavelengthNanometer'].values
# 元データの波長範囲内にあるHISUIバンドのみ抽出
data_wavs = data[0, :]
valid_indices = (hisui_wavs >= data_wavs.min()) & (hisui_wavs <= data_wavs.max())
target_wavs = hisui_wavs[valid_indices]

resampled_rows = []
resampled_rows.append(target_wavs) # 0行目は波長

# 平滑化データ(smoothed_data)をHISUI波長(target_wavs)に補間
for i in range(1, smoothed_data.shape[0]):
    y = smoothed_data[i, :]
    # 線形補間関数を作成
    f = interp1d(data_wavs, y, kind='linear')
    resampled_rows.append(f(target_wavs))

resampled_data = np.array(resampled_rows)

# --- プロット ---
# プロットする範囲の定義
plot_ranges = [
    (1000, 2500, "HISUI SWIR Range (1000-2500nm)"),
    (1500, 1800, "Zoom: 1500-1800nm"),
    (2100, 2400, "Zoom: 2100-2400nm")
]

ppm_labels = df_methane.columns[1:] # 0, 0.25, ... 5
colors = cm.viridis(np.linspace(0, 1, len(ppm_labels))) # 色の準備

fig, axes = plt.subplots(3, 1, figsize=(12, 18))

for ax, (start, end, title) in zip(axes, plot_ranges):
    # 波長範囲でフィルタリング
    mask = (resampled_data[0, :] >= start) & (resampled_data[0, :] <= end)
    x_plot = resampled_data[0, mask]
    
    # 各濃度のプロット
    for i in range(1, resampled_data.shape[0]):
        y_plot = resampled_data[i, mask]
        ax.plot(x_plot, y_plot, color=colors[i-1], linewidth=1)
        
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Wavelength [nm]", fontsize=12)
    ax.set_ylabel("Radiance / Value", fontsize=12)
    ax.grid(True, alpha=0.5)

# カラーバーの追加
sm = plt.cm.ScalarMappable(cmap=cm.viridis, norm=plt.Normalize(vmin=0, vmax=5))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), label='Methane Concentration (ppm)')

plt.savefig('hisui_methane_simulation.png')
plt.show()

In [ ]:
# --- プロット（3つの別図として表示） ---
plot_ranges = [
    (1000, 2500, "HISUI SWIR Range "),
    (1000, 1300, "Zoom: 1000-1300nm"),
    (2100, 2400, "Zoom: 2100-2400nm"),
]

ppm_labels = df_methane.columns[1:]  # 0, 0.25, ... 5
colors = cm.viridis(np.linspace(0, 1, len(ppm_labels)))  # 元コード踏襲

figs = []

for idx, (start, end, title) in enumerate(plot_ranges, start=1):
    fig, ax = plt.subplots(figsize=(12, 6))

    # 波長範囲でフィルタリング
    mask = (resampled_data[0, :] >= start) & (resampled_data[0, :] <= end)
    x_plot = resampled_data[0, mask]

    # 各濃度のプロット
    for i in range(1, resampled_data.shape[0]):
        y_plot = resampled_data[i, mask]
        ax.plot(x_plot, y_plot, color=colors[i-1], linewidth=1)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Wavelength [nm]", fontsize=12)
    ax.set_ylabel(r'Radiance [$W / (m^2 \cdot sr \cdot \mu m)$]', fontsize=12)
    ax.grid(True, alpha=0.5)

    # 各図ごとにカラーバーを付与（0〜5 ppmに正規化）
    sm = plt.cm.ScalarMappable(cmap=cm.viridis, norm=plt.Normalize(vmin=0, vmax=5))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label=r'H$_2$O Column Density($g/cm^2$)')

    fig.tight_layout()
    figs.append(fig)

# 個別に保存（不要ならコメントアウト）
for i, fig in enumerate(figs, start=1):
    fig.savefig(f"hisui_methane_simulation_{i}.eps", dpi=150)

plt.show()
